In [1]:
# Imports
import pandas as pd
import numpy as np

In [2]:
# Load the full corpus
file_path = r"data/combined_acl_arxiv_with_citations_final.csv"
df = pd.read_csv(file_path)
df.head()

,project_id,original_id,abstract,citation_count,publication_year,title,authors_str,source,original_id_clean
0,0,O02-2002,There is a need to measure word similarity whe...,14.0,2002,A Study on Word Similarity using Context Vecto...,"Chen, Keh-Jiann and\nYou, Jia-Ming",ACL,O02-2002
1,2,R13-1042,Thread disentanglement is the task of separati...,10.0,2013,"Headerless, Quoteless, but not Hopeless? Using...","Jamison, Emily and\nGurevych, Iryna",ACL,R13-1042
2,3,W05-0819,"In this paper, we describe a word alignment al...",15.0,2005,Aligning Words in {E}nglish-{H}indi Parallel C...,"Aswani, Niraj and\nGaizauskas, Robert",ACL,W05-0819
3,5,R13-1044,The paper 1 presents a rule-based approach to ...,2.0,2013,Recognizing semantic relations within {P}olish...,"K{\k{e}}dzia, Pawe{\l} and\nMaziarz, Marek",ACL,R13-1044
4,6,W05-0818,"In this paper we describe LIHLA, a lexical ali...",5.0,2005,{LIHLA}: Shared Task System Description,"Caseli, Helena M. and\nNunes, Maria G. V. an...",ACL,W05-0818


In [3]:
# Sort by publication_year ascending
df_sorted = df.sort_values('publication_year', ascending=True).reset_index(drop=True)
print(f"Total records: {len(df_sorted)}")
print(f"Year range: {df_sorted['publication_year'].min()} to {df_sorted['publication_year'].max()}")
print(f"\nYear distribution:")
print(df_sorted['publication_year'].value_counts().sort_index())
df_sorted.head(10)

Total records: 131772
Year range: 1952 to 2025

Year distribution:
publication_year
1952        6
1956        1
1957        1
1960        9
1961        7
        ...  
2021    10830
2022     9835
2023    11496
2024    20129
2025     6958
Name: count, Length: 63, dtype: int64


,project_id,original_id,abstract,citation_count,publication_year,title,authors_str,source,original_id_clean
0,48859,1952.earlymt-1.17,than in the procedures by which such informati...,1.0,1952,Problems of vocabulary frequency and distribution,"Bull, William E.",ACL,1952.earlymt-1.17
1,48766,1952.earlymt-1.2,"On November 30, 1950, WFL sent out a letter of...",0.0,1952,Present interest in mechanical translation,"Loomis, W. F.",ACL,1952.earlymt-1.2
2,48776,1952.earlymt-1.7,The problem of mechanical translation has thre...,0.0,1952,The structure of the problem of mechanical tra...,"Helmer, Olaf",ACL,1952.earlymt-1.7
3,48721,1952.earlymt-1.1,The attached memorandum on translation from on...,0.0,1952,Translation,"Weaver, Warren",ACL,1952.earlymt-1.1
4,48831,1952.earlymt-1.13,When I learned that I had been summoned to add...,3.0,1952,Word-by-word translation,"Oswald, Victor A.",ACL,1952.earlymt-1.13
5,49294,1952.earlymt-1.26,The following report was prepared immediately ...,9.0,1952,The conference on mechanical translation held ...,"Reynolds, Craig",ACL,1952.earlymt-1.26
6,63626,1956.earlymt-1.3,Certain postulates are posited as a basis for ...,0.0,1956,Organisation and Method in Mechanical Translat...,"Dostert, L. E.",ACL,1956.earlymt-1.3
7,63607,1957.earlymt-1.12,Lexical Search In recent studies of Machine Tr...,0.0,1957,The Requirements of Lexical Storage,"King, Gilbert W.",ACL,1957.earlymt-1.12
8,11003,1960.earlymt-nsmt.32,This paper presents a preliminary description ...,0.0,1960,Nestings Within the Prepositional Structure,"Zarechnak, Michael",ACL,1960.earlymt-nsmt.32
9,48737,1960.earlymt-fsmtw.6,WAYNE STATE UNIVERSITY PRESENTATION JANIOTIS M...,0.0,1960,Centro di Cibernetica di {M}ilano Presentation,"Ceccato, Silvio",ACL,1960.earlymt-fsmtw.6


In [9]:
# Temporal 80/10/10 split: <=2023 train, 2024 val, >=2025 test (no year overlap)
import json

# Calculate target counts for 80-10-10 split
total_records = len(df_sorted)
train_target = int(0.80 * total_records)
val_target = int(0.10 * total_records)
test_target = total_records - train_target - val_target

print(f"Target split counts:")
print(f"Train: {train_target} ({train_target/total_records*100:.2f}%)")
print(f"Val: {val_target} ({val_target/total_records*100:.2f}%)")
print(f"Test: {test_target} ({test_target/total_records*100:.2f}%)")
print()

# Strategy: Use years up to 2023 for train, 2024 for val, 2025 for test
# This ensures complete non-overlapping years

train_df = df_sorted[df_sorted['publication_year'] <= 2023].copy()
val_df = df_sorted[df_sorted['publication_year'] == 2024].copy()
test_df = df_sorted[df_sorted['publication_year'] >= 2025].copy()

print(f"{'='*60}")
print(f"FINAL SPLIT SUMMARY (Fully non-overlapping years):")
print(f"{'='*60}")
print(f"Train: {len(train_df):,} ({len(train_df)/total_records*100:.2f}%)")
print(f"       Years: {int(train_df['publication_year'].min())}-{int(train_df['publication_year'].max())}")
print(f"\nVal:   {len(val_df):,} ({len(val_df)/total_records*100:.2f}%)")
print(f"       Years: {int(val_df['publication_year'].min())}-{int(val_df['publication_year'].max())}")
print(f"\nTest:  {len(test_df):,} ({len(test_df)/total_records*100:.2f}%)")
print(f"       Years: {int(test_df['publication_year'].min())}-{int(test_df['publication_year'].max())}")
print(f"\nTotal: {len(train_df) + len(val_df) + len(test_df):,}")

# Verify no overlap
print(f"\n{'='*60}")
print(f"Verification - Fully non-overlapping years:")
print(f"{'='*60}")
print(f"Train: 1952-2023 (all years ≤ 2023)")
print(f"Val:   2024-2024 (only year 2024)")
print(f"Test:  2025-2025 (all years ≥ 2025)")
print(f"✓ No overlap guaranteed!")

print(f"\n{'='*60}")
print(f"Deviation from 80-10-10 target:")
print(f"{'='*60}")
print(f"Train: {len(train_df)-train_target:+,} records ({(len(train_df)/total_records*100 - 80):+.2f}%)")
print(f"Val:   {len(val_df)-val_target:+,} records ({(len(val_df)/total_records*100 - 10):+.2f}%)")
print(f"Test:  {len(test_df)-test_target:+,} records ({(len(test_df)/total_records*100 - 10):+.2f}%)")

Target split counts:
Train: 105417 (80.00%)
Val: 13177 (10.00%)
Test: 13178 (10.00%)

FINAL SPLIT SUMMARY (Fully non-overlapping years):
Train: 104,685 (79.44%)
       Years: 1952-2023

Val:   20,129 (15.28%)
       Years: 2024-2024

Test:  6,958 (5.28%)
       Years: 2025-2025

Total: 131,772

Verification - Fully non-overlapping years:
Train: 1952-2023 (all years ≤ 2023)
Val:   2024-2024 (only year 2024)
Test:  2025-2025 (all years ≥ 2025)
✓ No overlap guaranteed!

Deviation from 80-10-10 target:
Train: -732 records (-0.56%)
Val:   +6,952 records (+5.28%)
Test:  -6,220 records (-4.72%)


In [10]:
# Create JSON with project_ids
split_dict = {
    "train": train_df['project_id'].tolist(),
    "val": val_df['project_id'].tolist(),
    "test": test_df['project_id'].tolist()
}

# Save to JSON file
output_path = r"train_val_test_split.json"
with open(output_path, 'w') as f:
    json.dump(split_dict, f, indent=4)

print(f"✓ JSON file saved to: {output_path}")
print(f"\nJSON structure:")
print(f"  train: {len(split_dict['train'])} project_ids")
print(f"  val:   {len(split_dict['val'])} project_ids")
print(f"  test:  {len(split_dict['test'])} project_ids")
print(f"\nSample from each split:")
print(f"  train: {split_dict['train'][:5]}")
print(f"  val:   {split_dict['val'][:5]}")
print(f"  test:  {split_dict['test'][:5]}")

In [11]:
# Verify the JSON file content
with open(output_path, 'r') as f:
    loaded_data = json.load(f)

print("✓ JSON file verified successfully!")
print(f"\nFile contains {len(loaded_data)} keys: {list(loaded_data.keys())}")
print(f"\nFirst 10 project_ids from each split:")
print(f"  train: {loaded_data['train'][:10]}")
print(f"  val:   {loaded_data['val'][:10]}")
print(f"  test:  {loaded_data['test'][:10]}")

✓ JSON file verified successfully!

File contains 3 keys: ['train', 'val', 'test']

First 10 project_ids from each split:
  train: [48859, 48766, 48776, 48721, 48831, 49294, 63626, 63607, 11003, 48737]
  val:   [82537, 82538, 107336, 107444, 107374, 107367, 107430, 150044, 85819, 85740]
  test:  [74708, 86080, 120191, 75370, 76640, 142507, 142492, 142489, 142481, 142472]
